## 1. Tooth Detection and Segmentation
Process dental cases with the Tooth Segmentation + Recognition model.
This step outputs bounding-box images, mask overlays, and per-case JSON files.

Next step output used by Step 2:
- segmentation+recognition-dataset

In [1]:
# Step 1: Tooth Detection and Segmentation on raw_data (all cases)
import os
import json
import random
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# IMPORTANT for Windows/Jupyter: use non-interactive backend to avoid Tkinter issues
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ultralytics import YOLO
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    union = float(areaA + areaB - inter)
    return inter / union if union > 0 else 0.0


def get_device():
    import torch

    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using device: cuda | GPU: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        print("Using device: mps")
    else:
        device = "cpu"
        print("Using device: cpu")
    return device


def get_detectron2_device(device: str) -> str:
    # Detectron2 reliably supports cuda/cpu. If main device is mps, use cpu for Detectron2.
    if device == "cuda":
        return "cuda"
    if device == "mps":
        print("Detectron2 uses cpu because mps support is limited.")
        return "cpu"
    return "cpu"


def initialize_models(raw_data_dir: Path):
    device = get_device()
    detectron_device = get_detectron2_device(device)

    # Try common locations for model weights (supports both 'weights' and 'weights(model)')
    yolo_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_pano_20250319.pt",
    ]
    detectron_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_crop_20250424.pth",
    ]

    yolo_path = next((p for p in yolo_candidates if p.exists()), None)
    detectron_path = next((p for p in detectron_candidates if p.exists()), None)

    if yolo_path is None or detectron_path is None:
        checked = [str(p) for p in (yolo_candidates + detectron_candidates)]
        raise FileNotFoundError(
            "Model weights not found. Checked paths:\n" + "\n".join(checked)
        )

    print(f"YOLO weights: {yolo_path}")
    print(f"Detectron2 weights: {detectron_path}")

    yolo_model = YOLO(str(yolo_path))
    yolo_model.to(device)

    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.WEIGHTS = str(detectron_path)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.DEVICE = detectron_device
    detectron_predictor = DefaultPredictor(cfg)

    print("Models initialized successfully")
    return yolo_model, detectron_predictor


def process_panoramic_image(pano_img_rgb, yolo_model):
    results = yolo_model(pano_img_rgb, verbose=False)
    h, w = pano_img_rgb.shape[:2]
    all_masks = []

    for result in results:
        if result.masks is None or not hasattr(result.masks, "data"):
            continue

        temp = []
        num_masks = len(result.masks.xy) if hasattr(result.masks, "xy") else 0

        for i in range(num_masks):
            mask_matrix = result.masks.data[i].cpu().numpy()
            resized_mask = cv2.resize(
                mask_matrix.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
            ).astype(bool)

            bbox = result.boxes.xyxy[i].cpu().numpy().tolist()
            cls_id = int(result.boxes.cls[i].item())

            temp.append({
                "id": i,
                "matrix_resized": resized_mask,
                "confidence": float(result.boxes.conf[i].item()),
                "class_id": cls_id,
                "class_name": str(result.names[cls_id]),
                "bbox": bbox,
            })

        # Remove duplicates by IoU
        kept = []
        for m in temp:
            duplicated = False
            for k in kept:
                if compute_iou(m["bbox"], k["bbox"]) > 0.5:
                    if m["confidence"] > k["confidence"]:
                        kept.remove(k)
                        kept.append(m)
                    duplicated = True
                    break
            if not duplicated:
                kept.append(m)

        all_masks.extend(kept)

    return all_masks


def crop_and_segment_teeth(pano_img_rgb, mask_data_list, detectron_predictor, pad=20):
    outputs = []
    for data in mask_data_list:
        mask = data["matrix_resized"].astype(np.uint8) * 255
        x, y, w, h = cv2.boundingRect(mask)
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, pano_img_rgb.shape[1])
        y2 = min(y + h + pad, pano_img_rgb.shape[0])

        cropped = pano_img_rgb[y1:y2, x1:x2]
        pred = detectron_predictor(cropped)
        instances = pred["instances"].to("cpu")

        segments = []
        all_pixels = []
        if len(instances) > 0:
            masks = instances.pred_masks.numpy()
            scores = instances.scores.numpy()
            for idx, (seg_mask, score) in enumerate(zip(masks, scores)):
                rows, cols = np.where(seg_mask)
                pixel_coords = [(int(x1 + c), int(y1 + r)) for r, c in zip(rows, cols)]
                all_pixels.extend(pixel_coords)
                segments.append({
                    "mask_id": idx,
                    "score": float(score),
                    "num_pixels": len(pixel_coords),
                    "pixel_coordinates": pixel_coords,
                })

        outputs.append({
            "tooth_id": data["class_name"],
            "crop_coords": [int(x1), int(y1), int(x2), int(y2)],
            "num_segments": len(segments),
            "segments": segments,
            "pixel_coordinates": all_pixels,
            "total_pixels": len(all_pixels),
            "detectron_output": pred,
        })

    return outputs


def save_bbox_visualization(pano_img_rgb, list_of_masks, output_path):
    vis = pano_img_rgb.copy()
    for m in list_of_masks:
        x1, y1, x2, y2 = map(int, m["bbox"])
        label = str(m["class_name"])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    fig = plt.figure(figsize=(12, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Bounding Boxes with Tooth ID")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_mask_overlay(pano_img_rgb, list_seg_tooth, output_path, alpha=0.5):
    vis = pano_img_rgb.copy()

    for result in list_seg_tooth:
        x1, y1, x2, y2 = result["crop_coords"]
        pred = result["detectron_output"]
        instances = pred["instances"].to("cpu")

        if len(instances) == 0:
            continue

        masks = instances.pred_masks.numpy()
        for mask in masks:
            h, w = mask.shape
            color = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)

            color_mask = np.zeros((h, w, 3), dtype=np.uint8)
            color_mask[mask] = color

            crop = vis[y1:y2, x1:x2]
            blended = np.where(
                mask[:, :, None],
                cv2.addWeighted(crop, 1 - alpha, color_mask, alpha, 0),
                crop,
            )
            vis[y1:y2, x1:x2] = blended

    fig = plt.figure(figsize=(16, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Mask Overlay")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_results_json(case_num, list_of_masks, list_seg_tooth, output_dir):
    result = {
        "case_number": str(case_num),
        "num_teeth_detected": len(list_of_masks),
        "teeth_data": [],
    }

    for mask_data, seg_data in zip(list_of_masks, list_seg_tooth):
        result["teeth_data"].append({
            "tooth_id": mask_data["class_name"],
            "confidence": float(mask_data["confidence"]),
            "bbox": [float(v) for v in mask_data["bbox"]],
            "crop_coords": seg_data["crop_coords"],
            "num_segments": seg_data["num_segments"],
            "total_pixels": seg_data["total_pixels"],
            "pixel_coordinates": seg_data["pixel_coordinates"],
            "segments_detail": seg_data["segments"],
        })

    with open(output_dir / f"case_{case_num}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def process_case(case_dir, yolo_model, detectron_predictor, output_root):
    case_num = case_dir.name.replace("case ", "")
    png_path = case_dir / f"case_{case_num}.png"
    if not png_path.exists():
        png_files = sorted(case_dir.glob("*.png"))
        if not png_files:
            return False, "No PNG found"
        png_path = png_files[0]

    img_bgr = cv2.imread(str(png_path))
    if img_bgr is None:
        return False, "Image read failed"

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    list_of_masks = process_panoramic_image(img_rgb, yolo_model)
    if len(list_of_masks) == 0:
        return False, "No teeth detected"

    list_seg_tooth = crop_and_segment_teeth(img_rgb, list_of_masks, detectron_predictor)

    case_out = output_root / f"case {case_num}"
    case_out.mkdir(parents=True, exist_ok=True)

    save_bbox_visualization(img_rgb, list_of_masks, case_out / f"case_{case_num}_bounding_boxes.png")
    save_mask_overlay(img_rgb, list_seg_tooth, case_out / f"case_{case_num}_mask_overlay.png")
    save_results_json(case_num, list_of_masks, list_seg_tooth, case_out)

    return True, f"OK ({len(list_of_masks)} teeth)"


def run_inference_on_raw_dataset(max_cases=None):
    project_root = Path.cwd()
    # Adjusting paths relative to the current working directory
    raw_data_dir = project_root.parent / "data"
    cases_root = raw_data_dir / "500 cases with annotation"

    if not cases_root.exists():
        raise FileNotFoundError(f"Cases folder not found: {cases_root}")

    output_root = project_root / "segmentation+recognition-dataset"
    output_root.mkdir(parents=True, exist_ok=True)

    yolo_model, detectron_predictor = initialize_models(raw_data_dir)

    case_dirs = sorted(
        [d for d in cases_root.iterdir() if d.is_dir() and d.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    if max_cases is not None:
        case_dirs = case_dirs[:max_cases]
        print(f"Limiting execution to {max_cases} sample cases.")

    ok_count = 0
    fail_count = 0
    print(f"Processing cases: {len(case_dirs)}")

    for case_dir in tqdm(case_dirs, desc="raw_data cases", unit="case"):
        try:
            ok, msg = process_case(case_dir, yolo_model, detectron_predictor, output_root)
            if ok:
                ok_count += 1
            else:
                fail_count += 1
        except Exception as e:
            fail_count += 1
            print(f"Error in {case_dir.name}: {e}")

    summary = {
        "source": "data/500 cases with annotation-raw",
        "processed": ok_count,
        "failed": fail_count,
        "total": len(case_dirs),
    }

    with open(output_root / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nDone")
    print(f"Output folder: {output_root.resolve()}")
    print(json.dumps(summary, indent=2))


# RUN ONLY 5 SAMPLES FOR TESTING
run_inference_on_raw_dataset(max_cases=5)

# Uncomment the line below to run on ALL cases
# run_inference_on_raw_dataset()

d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cuda | GPU: NVIDIA GeForce RTX 4080 Laptop GPU
YOLO weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_pano_20250319.pt
Detectron2 weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_crop_20250424.pth


d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\fvcore\common\checkpoint.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=t

Models initialized successfully
Limiting execution to 5 sample cases.
Processing cases: 5


raw_data cases:   0%|          | 0/5 [00:00<?, ?case/s]d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
raw_data cases: 100%|██████████| 5/5 [00:31<00:00,  6.30s/case]


Done
Output folder: C:\Users\jaopi\Desktop\SP\phase2-1april\segmentation+recognition-dataset
{
  "source": "data/500 cases with annotation-raw",
  "processed": 5,
  "failed": 0,
  "total": 5
}


## 2. Map tooth positions to caries regions (JSON output)
Combined script that processes dental cases to:
1. Map tooth positions to caries regions (JSON output)
2. Generate visual alignment debug images (PNG output)

Data Sources:
- JSON: segmentation+recognition-dataset/case X/case_X_results.json
  Contains pixel_coordinates (list of [x, y]) for each tooth
- ROI Image: raw_data/500-roi/case_X.png
  Binary mask where non-zero = caries, 0 = normal

Output (single root folder):
- caries_mapping_output/case X/case_X_caries_mapping.json
- caries_mapping_output/case X/case_X_alignment_detailed.png
- caries_mapping_output/caries_mapping_results.csv (summary)

In [2]:
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


# =============================================================================
# ROI & JSON Loading Functions
# =============================================================================

def load_roi_image(roi_path: Path):
    """Load ROI mask and normalize to 2D grayscale array."""
    roi_img = cv2.imread(str(roi_path), cv2.IMREAD_UNCHANGED)
    if roi_img is None:
        raise FileNotFoundError(f"Could not load ROI image: {roi_path}")

    # Some PNGs are loaded as (H, W, 1); convert to strict 2D for downstream ops.
    if roi_img.ndim == 3:
        if roi_img.shape[2] == 1:
            roi_img = roi_img[:, :, 0]
        else:
            roi_img = cv2.cvtColor(roi_img, cv2.COLOR_BGR2GRAY)

    return roi_img


def load_tooth_json(json_path: Path):
    """Load tooth segmentation JSON with pixel coordinates."""
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# Caries Mapping Functions
# =============================================================================

def calculate_caries_overlap(roi_img, pixel_coordinates):
    """Calculate overlap between tooth pixels and caries region in ROI."""
    if not pixel_coordinates:
        return 0, 0, 0.0, []

    height, width = roi_img.shape[:2]
    caries_count = 0
    caries_coords = []
    valid_pixels = 0

    for coord in pixel_coordinates:
        x, y = coord[0], coord[1]

        if 0 <= x < width and 0 <= y < height:
            valid_pixels += 1
            if roi_img[y, x] > 0:
                caries_count += 1
                caries_coords.append([x, y])

    percentage = (caries_count / valid_pixels * 100) if valid_pixels > 0 else 0.0
    return caries_count, valid_pixels, percentage, caries_coords


def generate_caries_mapping(roi_img, tooth_data, case_num):
    """Generate per-tooth caries mapping results for one case."""
    results = []

    for tooth in tooth_data.get("teeth_data", []):
        tooth_id = tooth.get("tooth_id", "unknown")
        pixel_coords = tooth.get("pixel_coordinates", [])
        confidence = tooth.get("confidence", 0.0)

        caries_count, total_pixels, percentage, caries_coords = calculate_caries_overlap(
            roi_img, pixel_coords
        )

        results.append(
            {
                "case_number": int(case_num),
                "tooth_id": tooth_id,
                "confidence": float(confidence),
                "total_pixels": int(total_pixels),
                "caries_pixels": int(caries_count),
                "caries_percentage": round(float(percentage), 4),
                "has_caries": bool(caries_count > 0),
                "caries_coordinates": caries_coords,
            }
        )

    return results


# =============================================================================
# Visual Alignment Functions
# =============================================================================

def create_alignment_visualization(roi_img, tooth_data, alpha=0.5):
    """Create blended visualization: ROI background + tooth pixels overlay."""
    height, width = roi_img.shape[:2]

    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)
    roi_bgr = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    tooth_overlay = np.zeros((height, width, 3), dtype=np.uint8)

    colors = [
        (0, 255, 0),
        (0, 255, 255),
        (255, 255, 0),
        (0, 165, 255),
        (255, 0, 255),
        (128, 255, 0),
        (255, 128, 0),
        (0, 128, 255),
    ]

    tooth_count = 0
    total_pixels_drawn = 0
    out_of_bounds = 0

    for tooth in tooth_data.get("teeth_data", []):
        pixel_coords = tooth.get("pixel_coordinates", [])
        color = colors[tooth_count % len(colors)]
        tooth_count += 1

        for coord in pixel_coords:
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_overlay[y, x] = color
                total_pixels_drawn += 1
            else:
                out_of_bounds += 1

    tooth_mask = np.any(tooth_overlay > 0, axis=2)
    result = roi_bgr.copy()
    blended = cv2.addWeighted(roi_bgr, 1 - alpha, tooth_overlay, alpha, 0)
    result[tooth_mask] = blended[tooth_mask]
    strong_overlay = cv2.addWeighted(result, 0.7, tooth_overlay, 0.3, 0)
    result[tooth_mask] = strong_overlay[tooth_mask]

    return result, {
        "teeth_count": tooth_count,
        "pixels_drawn": total_pixels_drawn,
        "out_of_bounds": out_of_bounds,
        "roi_shape": (height, width),
    }


def create_detailed_alignment_image(roi_img, tooth_data):
    """Create 3-panel alignment image for QA/debug."""
    height, width = roi_img.shape[:2]
    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)

    roi_colored = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    caries_mask = roi_img > 0
    roi_colored[caries_mask] = [0, 0, 255]

    tooth_only = np.zeros((height, width, 3), dtype=np.uint8)
    for tooth in tooth_data.get("teeth_data", []):
        for coord in tooth.get("pixel_coordinates", []):
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_only[y, x] = [0, 255, 0]

    blended, stats = create_alignment_visualization(roi_img, tooth_data, alpha=0.6)

    tooth_mask = np.any(tooth_only > 0, axis=2)
    overlap_pixels = int(np.sum(tooth_mask & caries_mask))
    caries_pixels = int(np.sum(caries_mask))
    tooth_pixels = int(np.sum(tooth_mask))

    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(roi_colored, "ROI (Red=Caries)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(tooth_only, "Teeth (Green)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(blended, "ALIGNMENT CHECK", (20, 60), font, 1.5, (0, 255, 255), 3)

    stats_text = [
        f"Teeth: {stats['teeth_count']}",
        f"Tooth pixels: {tooth_pixels:,}",
        f"Caries pixels: {caries_pixels:,}",
        f"Overlap: {overlap_pixels:,}",
    ]
    y_offset = 120
    for text in stats_text:
        cv2.putText(blended, text, (20, y_offset), font, 1.0, (255, 255, 0), 2)
        y_offset += 40

    scale = 0.5
    roi_small = cv2.resize(roi_colored, None, fx=scale, fy=scale)
    tooth_small = cv2.resize(tooth_only, None, fx=scale, fy=scale)
    blended_small = cv2.resize(blended, None, fx=scale, fy=scale)
    combined = np.hstack([roi_small, tooth_small, blended_small])

    overlap_stats = {
        "overlap_pixels": overlap_pixels,
        "caries_pixels": caries_pixels,
        "tooth_pixels": tooth_pixels,
    }
    return combined, stats, overlap_stats


# =============================================================================
# Pipeline Functions for Current Project Structure
# =============================================================================

def parse_case_num_from_json(json_path: Path):
    """Extract numeric case id from filename: case_123_results.json -> 123."""
    stem = json_path.stem
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected JSON filename format: {json_path.name}")
    return int(parts[1])


def collect_case_records(segmentation_root: Path, roi_root: Path):
    """Collect case records from Step 1 output and matching ROI paths."""
    records = []

    case_dirs = sorted(
        [p for p in segmentation_root.iterdir() if p.is_dir() and p.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    for case_dir in case_dirs:
        json_candidates = sorted(case_dir.glob("case_*_results.json"))
        if not json_candidates:
            continue

        json_path = json_candidates[0]
        case_num = parse_case_num_from_json(json_path)
        roi_path = roi_root / f"case_{case_num}.png"

        records.append(
            {
                "case_number": case_num,
                "json_path": json_path,
                "roi_path": roi_path,
            }
        )

    return records


def process_single_case(record, output_root: Path):
    """Process one case and save both JSON + alignment image in one case folder."""
    case_num = record["case_number"]
    json_path = record["json_path"]
    roi_path = record["roi_path"]

    if not json_path.exists():
        return False, "JSON not found", None
    if not roi_path.exists():
        return False, f"ROI not found: {roi_path.name}", None

    roi_img = load_roi_image(roi_path)
    tooth_data = load_tooth_json(json_path)
    caries_results = generate_caries_mapping(roi_img, tooth_data, case_num)

    case_output_dir = output_root / f"case {case_num}"
    case_output_dir.mkdir(parents=True, exist_ok=True)

    caries_json_path = case_output_dir / f"case_{case_num}_caries_mapping.json"
    with open(caries_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "case_number": int(case_num),
                "teeth_caries_data": caries_results,
            },
            f,
            indent=2,
        )

    combined_img, stats, overlap_stats = create_detailed_alignment_image(roi_img, tooth_data)
    alignment_img_path = case_output_dir / f"case_{case_num}_alignment_detailed.png"
    cv2.imwrite(str(alignment_img_path), combined_img)

    teeth_with_caries = sum(1 for r in caries_results if r["has_caries"])
    msg = f"{len(caries_results)} teeth, {teeth_with_caries} with caries"

    case_stats = {
        "stats": stats,
        "overlap_stats": overlap_stats,
        "caries_results": caries_results,
    }
    return True, msg, case_stats


def run_caries_mapping_pipeline(
    segmentation_root: Path,
    roi_root: Path,
    output_root: Path,
    max_cases=None,
):
    """Run mapping pipeline and keep all outputs under one root folder."""
    output_root.mkdir(parents=True, exist_ok=True)

    records = collect_case_records(segmentation_root, roi_root)
    if not records:
        raise FileNotFoundError(
            "No case JSON files found. Check segmentation_root path and Step 1 outputs."
        )

    if max_cases is not None:
        records = records[:max_cases]
        print(f"Limiting execution to {max_cases} sample cases.")

    summary_stats = {
        "total_cases": len(records),
        "processed_cases": 0,
        "failed_cases": 0,
        "total_teeth": 0,
        "teeth_with_caries": 0,
    }
    all_caries_results = []

    print("=" * 70)
    print("Dental Caries Mapping Pipeline")
    print("=" * 70)
    print(f"Segmentation JSON source: {segmentation_root}")
    print(f"ROI source: {roi_root}")
    print(f"Output root: {output_root}")
    print("=" * 70)

    pbar = tqdm(records, desc="Processing cases", unit="case")
    for record in pbar:
        case_num = record["case_number"]
        try:
            success, msg, payload = process_single_case(record, output_root=output_root)

            if success and payload:
                case_results = payload["caries_results"]
                summary_stats["processed_cases"] += 1
                summary_stats["total_teeth"] += len(case_results)
                summary_stats["teeth_with_caries"] += sum(
                    1 for r in case_results if r["has_caries"]
                )

                for r in case_results:
                    csv_result = {k: v for k, v in r.items() if k != "caries_coordinates"}
                    all_caries_results.append(csv_result)

                pbar.set_postfix_str(f"case {case_num}: {msg}")
            else:
                summary_stats["failed_cases"] += 1
                pbar.set_postfix_str(f"case {case_num}: FAIL - {msg}")

        except Exception as e:
            summary_stats["failed_cases"] += 1
            pbar.set_postfix_str(f"case {case_num}: ERROR - {str(e)[:40]}")

    if all_caries_results:
        df = pd.DataFrame(all_caries_results)
        csv_path = output_root / "caries_mapping_results.csv"
        df.to_csv(csv_path, index=False)
        print(f"\nResults CSV saved to: {csv_path}")

        summary_df = (
            df.groupby(["case_number"]).agg(
                {
                    "tooth_id": "count",
                    "caries_pixels": "sum",
                    "has_caries": "sum",
                    "caries_percentage": "mean",
                }
            ).rename(
                columns={
                    "tooth_id": "total_teeth",
                    "has_caries": "teeth_with_caries",
                    "caries_percentage": "avg_caries_percentage",
                }
            )
        )
        summary_csv_path = output_root / "caries_summary_by_case.csv"
        summary_df.to_csv(summary_csv_path)
        print(f"Summary CSV saved to: {summary_csv_path}")

    print("\n" + "=" * 70)
    print("Processing Complete")
    print("=" * 70)
    print(f"Total cases found: {summary_stats['total_cases']}")
    print(f"Processed: {summary_stats['processed_cases']}")
    print(f"Failed: {summary_stats['failed_cases']}")
    print(f"Total teeth analyzed: {summary_stats['total_teeth']}")
    print(f"Teeth with caries: {summary_stats['teeth_with_caries']}")
    if summary_stats["total_teeth"] > 0:
        pct = summary_stats["teeth_with_caries"] / summary_stats["total_teeth"] * 100
        print(f"Caries prevalence: {pct:.2f}%")
    print("=" * 70)

    return all_caries_results, summary_stats


# =============================================================================
# Notebook Runner (Current Project Environment)
# =============================================================================

project_root = Path.cwd()

# Step 1 output folder (refactored pipeline)
segmentation_root = project_root / "segmentation+recognition-dataset"
if not segmentation_root.exists():
    raise FileNotFoundError(
        f"Segmentation output folder not found: {segmentation_root}. Run Step 1 first."
    )

# ROI folder from raw_data
roi_root = project_root.parent / "data" / "500-roi"
if not roi_root.exists():
    raise FileNotFoundError(
        f"ROI folder not found: {roi_root}."
    )

# Single output root as requested
output_root = project_root / "caries_mapping_output"

# RUN ONLY 5 SAMPLES FOR TESTING
all_caries_results, summary_stats = run_caries_mapping_pipeline(
    segmentation_root=segmentation_root,
    roi_root=roi_root,
    output_root=output_root,
    max_cases=5
)

# Uncomment the below block to run on ALL cases
# all_caries_results, summary_stats = run_caries_mapping_pipeline(
#     segmentation_root=segmentation_root,
#     roi_root=roi_root,
#     output_root=output_root,
# )

Limiting execution to 5 sample cases.
Dental Caries Mapping Pipeline
Segmentation JSON source: c:\Users\jaopi\Desktop\SP\phase2-1april\segmentation+recognition-dataset
ROI source: c:\Users\jaopi\Desktop\SP\data\500-roi
Output root: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output


Processing cases: 100%|██████████| 5/5 [00:08<00:00,  1.77s/case, case 5: 23 teeth, 7 with caries]


Results CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_mapping_results.csv
Summary CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_summary_by_case.csv

Processing Complete
Total cases found: 5
Processed: 5
Failed: 0
Total teeth analyzed: 145
Teeth with caries: 33
Caries prevalence: 22.76%


## 3. PCA-based tooth alignment + 3-surface classification (Visualization Output)

This module extends the pipeline by applying PCA-based alignment
to each tooth and classifying caries into 3 surfaces:
- Occlusal
- Mesial
- Distal

It also generates visualization outputs for debugging and interpretation.

Output Structure:
- PCA_Output/case X/
    <!-- - case_X_pca_alignment.png -->
    - case_X_pca_results.json

## 4. Visualization of PCA-Aligned Teeth and Caries Surfaces

This module provides visualization functions for each tooth after PCA alignment.
It supports:

Showing aligned tooth polygon
Displaying Occlusal / Mesial / Distal zones
Plotting caries pixels and centroid
Drawing bounding box for reference
Generating grid layout for all teeth in a case

Output can be saved as .png for reporting or debugging.


In [ ]:
# PCA & Surface Classification [VERSION 4.5.1 — week7 + 40/20/40 zone split]
# Key changes from v4.x series:
#  1. PCA Rule 1: pick eigenvector with largest |Y| (not eigenvalue) -> fixes square molars
#  2. PCA Rule 3: enforce horizontal axis direction by quadrant -> CONSISTENT M/D direction
#  3. PCA Rule 4: clamp |angle| > 45 deg to 0 (safety net for bad masks)
#  4. Classification: X-thirds (center 1/3 = Occlusal, sides = M/D)
#     Q1/Q4: left=Distal, center=Occlusal, right=Mesial  (per week7 FDI anatomy)
#     Q2/Q3: left=Mesial, center=Occlusal, right=Distal

import os, json, math
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

SEG_DIR    = r"C:\Users\jaopi\Desktop\SP\week2-Tooth Detection & Segmentation\500-segmentation+recognition"
CARIES_DIR = r"C:\Users\jaopi\Desktop\SP\week3-Caries-to-Tooth Mapping\dental_analysis_output"
OUT_DIR    = "PCA_Output"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_TILT_DEG      = 45.0   # clamp extreme PCA angles (bad tooth masks)
MIN_CLUSTER_SIZE  = 15     # noise removal: drop clusters smaller than this
LEFT_BOUND  = 0.40   # X-split: Left zone  0.00-0.40 (wider = fewer M/D->Occ spillover)
RIGHT_BOUND = 0.60   # X-split: Right zone 0.60-1.00 | Center = 0.40-0.60 (Occlusal)

SURFACE_COLORS = {"Occlusal": "#E74C3C", "Mesial": "#3498DB",
                  "Distal": "#27AE60", "Other": "#2ECC71", -1: "#95A5A6"}

def is_upper_jaw(tid): return int(str(tid)[0]) in [1, 2]
def get_quadrant(tid):  return int(str(tid)[0])

def load_seg(case_id):
    path = os.path.join(SEG_DIR, f"case {case_id}", f"case_{case_id}_results.json")
    if not os.path.exists(path): print(f"[ERROR] SEG: {path}"); return None
    with open(path) as f: return json.load(f)

def load_caries(case_id):
    path = os.path.join(CARIES_DIR, f"case {case_id}", f"case_{case_id}_caries_mapping.json")
    if not os.path.exists(path): print(f"[ERROR] CARIES: {path}"); return None
    with open(path) as f: return json.load(f)

def build_seg_map(seg_data):
    return {str(t["tooth_id"]): t.get("pixel_coordinates", []) for t in seg_data.get("teeth_data", [])}

def get_caries_list(d): return d.get("teeth_caries_data", [])
def compute_centroid(p): a=np.array(p,dtype=np.float64); return float(np.mean(a[:,0])),float(np.mean(a[:,1]))

def get_bbox(pts):
    p=np.array(pts,dtype=np.float64); mn,mx=np.min(p,0),np.max(p,0)
    return mn[0],mn[1],mx[0]-mn[0],mx[1]-mn[1]


# =============================================================================
# NOISE REMOVAL (from week7)
# =============================================================================
def remove_small_clusters(caries_pts, min_cluster=MIN_CLUSTER_SIZE):
    if len(caries_pts) < min_cluster:
        return caries_pts
    pts = np.array(caries_pts, dtype=np.int32)
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    pad = 2
    w = x_max - x_min + 1 + 2*pad
    h = y_max - y_min + 1 + 2*pad
    mask = np.zeros((h, w), dtype=np.uint8)
    shifted = pts - np.array([x_min - pad, y_min - pad])
    mask[shifted[:,1], shifted[:,0]] = 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros_like(mask)
    for lbl in range(1, n_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_cluster:
            keep[labels == lbl] = 255
    ys, xs = np.where(keep > 0)
    if len(xs) == 0: return caries_pts
    return np.column_stack([xs + x_min - pad, ys + y_min - pad]).astype(np.float64)


# =============================================================================
# PCA  — 4-Rule Orientation (ported from week7 multi_zone_classifier.py)
# =============================================================================
def perform_pca(points, tooth_id):
    """
    4-Rule PCA orientation (week7 reference):
      Rule 1 – Verticality: pick eigenvector with larger |Y| as vertical axis
      Rule 2 – Occlusal/Apical: flip vertical axis to point toward crown
      Rule 3 – Mesial/Distal: enforce horizontal axis direction
               Q1/Q4: horizontal_axis[0] > 0 (Mesial is +X = right)
               Q2/Q3: horizontal_axis[0] < 0 (Mesial is -X = left)
      Rule 4 – Angle clamp: if |angle| > MAX_TILT_DEG, use 0 (safety net)
    Returns: (mean, rotation_angle_rad, clamped_bool)
    """
    pts = np.array(points, dtype=np.float64)
    mean = np.mean(pts, axis=0)
    centered = pts - mean

    # cv2.PCACompute for consistency with week7
    _, eigvecs = cv2.PCACompute(centered.astype(np.float32), mean=None)
    ev0 = eigvecs[0].astype(np.float64)  # largest variance
    ev1 = eigvecs[1].astype(np.float64)  # second

    # Rule 1: Verticality — pick eigenvector with larger |Y| as vertical axis
    if abs(ev0[1]) >= abs(ev1[1]):
        vertical_axis   = ev0.copy()
        horizontal_axis = ev1.copy()
    else:
        vertical_axis   = ev1.copy()   # square molar: ev1 is more vertical
        horizontal_axis = ev0.copy()

    # Rule 2: Occlusal/Apical direction
    upper = is_upper_jaw(tooth_id)
    if upper:
        if vertical_axis[1] < 0: vertical_axis = -vertical_axis   # crown faces +Y (down)
    else:
        if vertical_axis[1] > 0: vertical_axis = -vertical_axis   # crown faces -Y (up)

    # Rule 3: Mesial/Distal direction
    # Q1/Q4 (patient right, image left): midline is to the RIGHT -> Mesial at +X
    # Q2/Q3 (patient left, image right): midline is to the LEFT  -> Mesial at -X
    quadrant = get_quadrant(tooth_id)
    if quadrant in [1, 4]:
        if horizontal_axis[0] < 0: horizontal_axis = -horizontal_axis  # ensure +X
    else:
        if horizontal_axis[0] > 0: horizontal_axis = -horizontal_axis  # ensure -X

    # Compute rotation angle
    angle_from_x = math.atan2(vertical_axis[1], vertical_axis[0])
    rot = math.pi / 2 - angle_from_x
    while rot >  math.pi: rot -= 2 * math.pi
    while rot < -math.pi: rot += 2 * math.pi

    # Rule 4: Angle clamp
    clamped = False
    if abs(math.degrees(rot)) > MAX_TILT_DEG:
        rot = 0.0
        clamped = True

    return mean, rot, clamped


def rotate(pts, center, angle):
    p = np.array(pts, dtype=np.float64) - center
    c, s = np.cos(angle), np.sin(angle)
    return np.dot(p, np.array([[c,-s],[s,c]]).T) + center


# =============================================================================
# CLASSIFICATION  v4.5 — X-thirds dominant zone (week7 approach)
# =============================================================================
def classify_surface_full(tid, tooth_pts, caries_pts):
    """
    FDI anatomy reference (after 4-rule PCA, horizontal axis enforced):
      Q1/Q4 (image left):  Left third = Distal | Center = Occlusal | Right third = Mesial
      Q2/Q3 (image right): Left third = Mesial | Center = Occlusal | Right third = Distal

    Dominant zone (most pixels) determines the prediction.
    """
    # Step 1: Noise removal
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    # Step 2: PCA alignment (4-rule)
    center, angle, clamped = perform_pca(tooth_pts, tid)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    rel_xs = np.clip((caries_rot[:,0] - x) / w, 0.0, 1.0)
    n_pts  = len(rel_xs)

    # Step 3: X-thirds zone assignment
    # After Rule 3 enforcement, horizontal axis is consistent per quadrant:
    #   Q1/Q4: +X is toward Mesial -> right third = Mesial, left third = Distal
    #   Q2/Q3: -X is toward Mesial -> left third = Mesial, right third = Distal
    quadrant = get_quadrant(tid)
    if quadrant in [1, 4]:
        # Q1/Q4: low x = Distal, center = Occlusal, high x = Mesial
        d_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        m_mask = rel_xs > RIGHT_BOUND
    else:
        # Q2/Q3: low x = Mesial, center = Occlusal, high x = Distal
        m_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        d_mask = rel_xs > RIGHT_BOUND

    m_count = int(np.sum(m_mask))  # Mesial
    c_count = int(np.sum(c_mask))  # Occlusal (center)
    d_count = int(np.sum(d_mask))  # Distal

    # Step 4: Dominant zone wins
    vote_map = {"Mesial": m_count, "Occlusal": c_count, "Distal": d_count}
    winner   = max(vote_map, key=vote_map.get)

    vote_fractions = {k: round(v/max(n_pts,1), 4) for k,v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    return winner, float(math.degrees(angle)), vote_fractions


def compute_caries_stats(caries_pts, tooth_pts):
    cp, tp = len(caries_pts), len(tooth_pts)
    return cp, (cp/tp*100) if tp else 0


def visualize_tooth_with_zones(tooth_pts, caries_pts, tooth_id, classification,
                                vote_fractions=None, save_path=None):
    tooth_pts  = np.array(tooth_pts,  dtype=np.float64)
    caries_pts = np.array(caries_pts, dtype=np.float64)
    x, y = np.min(tooth_pts, axis=0); w, h = np.ptp(tooth_pts, axis=0)
    fig, ax = plt.subplots(figsize=(6, 6))
    q = get_quadrant(tooth_id)
    # Q1/Q4: left=Distal, center=Occlusal, right=Mesial
    # Q2/Q3: left=Mesial, center=Occlusal, right=Distal
    left_col  = SURFACE_COLORS["Distal"]  if q in [1,4] else SURFACE_COLORS["Mesial"]
    right_col = SURFACE_COLORS["Mesial"] if q in [1,4] else SURFACE_COLORS["Distal"]
    t1 = x + w*LEFT_BOUND; t2 = x + w*RIGHT_BOUND
    ax.fill([x,t1,t1,x],          [y,y,y+h,y+h], alpha=0.18, color=left_col)
    ax.fill([t1,t2,t2,t1],         [y,y,y+h,y+h], alpha=0.18, color=SURFACE_COLORS["Occlusal"])
    ax.fill([t2,x+w,x+w,t2],       [y,y,y+h,y+h], alpha=0.18, color=right_col)
    ax.scatter(tooth_pts[:,0], tooth_pts[:,1], c="gray", s=2, alpha=0.4)
    color = SURFACE_COLORS.get(classification, "#95A5A6")
    if len(caries_pts)>0: ax.scatter(caries_pts[:,0],caries_pts[:,1],c=color,s=10,zorder=5)
    cx,cy = compute_centroid(caries_pts)
    ax.plot(cx,cy,"*",markersize=14,color="gold",zorder=6)
    ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,ls="--",lw=1))
    title = f"Tooth {tooth_id} Q{q} — {classification} (v4.5)"
    if vote_fractions:
        v=vote_fractions; title+=f"\nOcc={v.get('Occlusal',0):.2f} M={v.get('Mesial',0):.2f} D={v.get('Distal',0):.2f}"
    ax.set_title(title,fontsize=9); ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig)
    return fig


def process_case(case_id):
    print(f"\n=== CASE {case_id} ===")
    seg_data    = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None: return
    seg_map     = build_seg_map(seg_data)
    caries_list = get_caries_list(caries_data)
    results = {"case_number": case_id, "teeth_data": []}
    for tooth in caries_list:
        tid        = str(tooth["tooth_id"])
        caries_pts = tooth.get("caries_coordinates", [])
        tooth_pts  = seg_map.get(tid, [])
        if len(caries_pts)==0 or len(tooth_pts)<10: continue
        surface,angle,vf = classify_surface_full(tid, tooth_pts, caries_pts)
        pixels,pct = compute_caries_stats(caries_pts, tooth_pts)
        results["teeth_data"].append({
            "tooth_id": tid, "has_caries": True,
            "confidence": tooth.get("confidence",0),
            "caries_position_detail": surface,
            "predicted_surface_fine": surface,
            "vote_fractions": vf,
            "rotation_angle": round(angle,2),
            "tooth_coordinates": tooth_pts,
            "caries_coordinates": caries_pts,
            "caries_pixels": pixels,
            "caries_percentage": round(pct,4)
        })
    case_dir = os.path.join(OUT_DIR, f"case_{case_id}")
    os.makedirs(case_dir, exist_ok=True)
    with open(os.path.join(case_dir, f"case_{case_id}.json"), "w") as f:
        json.dump(results, f, indent=4)
    print(f"[DONE] saved -> case_{case_id}.json")


def process_case_visual(case_id):
    case_dir=Path(OUT_DIR)/f"case_{case_id}"; jp=case_dir/f"case_{case_id}.json"
    if not jp.exists(): print(f"[SKIP] Case {case_id}"); return
    with open(jp) as f: data=json.load(f)
    for tooth in data["teeth_data"]:
        tid=tooth["tooth_id"]; tp=tooth["tooth_coordinates"]; cp=tooth["caries_coordinates"]
        c,a,_ = perform_pca(tp,tid); tr=rotate(tp,c,a); cr=rotate(np.array(cp,dtype=np.float64),c,a)
        td=case_dir/f"tooth_{tid}"; td.mkdir(parents=True,exist_ok=True)
        visualize_tooth_with_zones(tr,cr,tid,tooth["caries_position_detail"],
            vote_fractions=tooth.get("vote_fractions",{}),
            save_path=td/f"tooth_{tid}_visual.png")
    print(f"[DONE] visuals -> Case {case_id}")


for cid in range(1, 501):
    process_case(cid)
    process_case_visual(cid)


## XML Parse Coverage
เช็คว่า xml ที่แปลงมาเป็น GT ใช้ได้จริง


In [10]:
from pathlib import Path
import xml.etree.ElementTree as ET

# =========================================================
# Namespace
# =========================================================
AIM_NS = "gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM"
ISO_NS = "uri:iso.org:21090"
NS = {"aim": AIM_NS, "iso": ISO_NS}

# =========================================================
# Surface map
# =========================================================
SNODENT_SURFACE_MAP = {
    "144414D": "Occlusal",
    "146014D": "Distal",
    "145374D": "Mesial",
    "144474D": "Occlusal",
    "146074D": "Distal",
    "145434D": "Mesial",
}

DISPLAY_NAME_TO_SURFACE = {
    "Occlusal surface": "Occlusal",
    "Occlusal Surface": "Occlusal",
    "Distal Surface": "Distal",
    "Distal surface": "Distal",
    "Mesial Surface": "Mesial",
    "Mesial surface": "Mesial",
}

# =============================================================================
# SNODENT Code → FDI Tooth ID Mapping
# =============================================================================
# Key = SNODENT code from XML <ImagingPhysicalEntityCharacteristic>
# Value = FDI two-digit tooth number
#
# FDI Quadrants:
#   Q1 (11–18): Upper Right    Q2 (21–28): Upper Left
#   Q3 (31–38): Lower Left     Q4 (41–48): Lower Right

# =========================================================
# Tooth fallback map
# =========================================================
SNODENT_TO_FDI = {
    # ===================== UPPER RIGHT (Quadrant 1) =====================
    "161006D": "11",  # Permanent upper right central incisor tooth
    "160842D": "12",  # Permanent upper right lateral incisor tooth
    "160288D": "13",  # Permanent upper right canine tooth
    "161286D": "14",  # Permanent upper right first premolar tooth
    "160450D": "15",  # Permanent upper right second premolar tooth
    "160770D": "16",  # Permanent upper right first molar tooth       # NOTE: also used for lower in some datasets
    "161204D": "17",  # Permanent upper right second molar tooth
    "160618D": "18",  # Permanent upper right third molar tooth

    # ===================== UPPER LEFT (Quadrant 2) =====================
    "160194D": "21",  # Permanent upper left central incisor tooth
    "160132D": "22",  # Permanent upper left lateral incisor tooth
    "160506D": "23",  # Permanent upper left canine tooth
    "161340D": "24",  # Permanent upper left first premolar tooth
    "160682D": "25",  # Permanent upper left second premolar tooth
    "161074D": "26",  # Permanent upper left first molar tooth
    "160386D": "27",  # Permanent upper left second molar tooth
    "160922D": "28",  # Permanent upper left third molar tooth

    # ===================== LOWER LEFT (Quadrant 3) =====================
    "161136D": "31",  # Permanent lower left central incisor tooth
    "160556D": "32",  # Permanent lower left lateral incisor tooth
    "160068D": "33",  # Permanent lower left canine tooth
    "160326D": "34",  # Permanent lower left first premolar tooth
    "161248D": "35",  # Permanent lower left second premolar tooth
    "160730D": "36",  # Permanent lower left first molar tooth
    "161166D": "37",  # Permanent lower left second molar tooth
    "160580D": "38",  # Permanent lower left third molar tooth

    # ===================== LOWER RIGHT (Quadrant 4) =====================
    "160964D": "41",  # Permanent lower right central incisor tooth
    "160350D": "42",  # Permanent lower right lateral incisor tooth
    "160894D": "43",  # Permanent lower right canine tooth
    "160230D": "44",  # Permanent lower right first premolar tooth
    "161412D": "45",  # Permanent lower right second premolar tooth
    "160770D": "46",  # Permanent lower right first molar tooth       # CAUTION: shared code
    "161102D": "47",  # Permanent lower right second molar tooth
    "160488D": "48",  # Permanent lower right third molar tooth
}

# =========================================================
# Helpers
# =========================================================
def _get_display_name(element):
    dn = element.find("iso:displayName", NS)
    if dn is not None:
        return dn.get("value", "")
    return ""

def snodent_display_to_fdi(display_name):

    dn = display_name.lower()

    if "upper" in dn and "right" in dn:
        quadrant = 1
    elif "upper" in dn and "left" in dn:
        quadrant = 2
    elif "lower" in dn and "left" in dn:
        quadrant = 3
    elif "lower" in dn and "right" in dn:
        quadrant = 4
    else:
        return ""

    if "central incisor" in dn:
        pos = 1
    elif "lateral incisor" in dn:
        pos = 2
    elif "canine" in dn:
        pos = 3
    elif "first premolar" in dn:
        pos = 4
    elif "second premolar" in dn:
        pos = 5
    elif "first molar" in dn:
        pos = 6
    elif "second molar" in dn:
        pos = 7
    elif "third molar" in dn:
        pos = 8
    else:
        return ""

    return f"{quadrant}{pos}"

# =========================================================
# XML parser
# =========================================================
def parse_aim_xml(xml_path):

    try:
        tree = ET.parse(xml_path)
    except:
        return None

    root = tree.getroot()

    annotations = root.find("aim:imageAnnotations", NS)
    if annotations is None:
        return None

    ann = annotations.find("aim:ImageAnnotation", NS)
    if ann is None:
        return None

    tooth = ""
    surface = ""

    phys_coll = ann.find("aim:imagingPhysicalEntityCollection", NS)

    if phys_coll is not None:

        entity = phys_coll.find("aim:ImagingPhysicalEntity", NS)

        if entity is not None:

            char_coll = entity.find(
                "aim:imagingPhysicalEntityCharacteristicCollection", NS
            )

            if char_coll is not None:

                chars = char_coll.findall(
                    "aim:ImagingPhysicalEntityCharacteristic", NS
                )

                for ch in chars:

                    q_idx_el = ch.find("aim:questionIndex", NS)
                    q_idx = q_idx_el.get("value", "") if q_idx_el is not None else ""

                    tc = ch.find("aim:typeCode", NS)
                    if tc is None:
                        continue

                    code = tc.get("code", "")
                    display = _get_display_name(tc)

                    if q_idx == "0":

                        tooth = snodent_display_to_fdi(display)

                        if not tooth:
                            tooth = SNODENT_TO_FDI.get(code, "")

                    elif q_idx == "1":

                        surface = SNODENT_SURFACE_MAP.get(code, "")

                        if not surface:
                            surface = DISPLAY_NAME_TO_SURFACE.get(display, "")

    return {
        "tooth": tooth,
        "surface": surface
    }

# =========================================================
# COVERAGE CHECK
# =========================================================
base_path = Path("../data/500 cases with annotation")

ok = 0
missing_tooth = 0
missing_surface = 0
failed = 0

for case_folder in sorted(base_path.glob("case *")):

    for xml_file in case_folder.glob("*.xml"):

        parsed = parse_aim_xml(str(xml_file))

        if parsed is None:
            failed += 1
            continue

        if not parsed["tooth"]:
            missing_tooth += 1

        elif not parsed["surface"]:
            missing_surface += 1

        else:
            ok += 1

print("========== PARSE COVERAGE ==========")
print("OK lesions      :", ok)
print("Missing tooth   :", missing_tooth)
print("Missing surface :", missing_surface)
print("Failed XML      :", failed)

========== PARSE COVERAGE ==========
OK lesions      : 1979
Missing tooth   : 0
Missing surface : 0
Failed XML      : 0


## 5.Evaluation

parse XML ground truth + parse prediction refactor mapping + parser 
ทำ evaluation:
Accuracy
Precision
Recall
F1-score
Confusion Matrix

In [6]:
# =========================================================
# EVALUATION  — Self-Contained (includes XML parser)
# Run this cell independently; no need to run Cell 7 first.
# =========================================================

from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# XML Parser (from Cell 7 — embedded for self-contained run)
# =========================================================
AIM_NS = "gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM"
ISO_NS = "uri:iso.org:21090"
NS = {"aim": AIM_NS, "iso": ISO_NS}

SNODENT_SURFACE_MAP = {
    "144414D": "Occlusal",
    "146014D": "Distal",
    "145374D": "Mesial",
    "144474D": "Occlusal",
    "146074D": "Distal",
    "145434D": "Mesial",
}

DISPLAY_NAME_TO_SURFACE = {
    "Occlusal surface": "Occlusal",
    "Occlusal Surface": "Occlusal",
    "Distal Surface":   "Distal",
    "Distal surface":   "Distal",
    "Mesial Surface":   "Mesial",
    "Mesial surface":   "Mesial",
}

SNODENT_TO_FDI = {
    # Upper Right Q1
    "161006D": "11", "160842D": "12", "160288D": "13", "161286D": "14",
    "160450D": "15", "160770D": "16", "161204D": "17", "160618D": "18",
    # Upper Left Q2
    "160194D": "21", "160132D": "22", "160506D": "23", "161340D": "24",
    "160682D": "25", "161074D": "26", "160386D": "27", "160922D": "28",
    # Lower Left Q3
    "161136D": "31", "160556D": "32", "160068D": "33", "160326D": "34",
    "161248D": "35", "160730D": "36", "161166D": "37", "160580D": "38",
    # Lower Right Q4
    "160964D": "41", "160350D": "42", "160894D": "43", "160230D": "44",
    "161412D": "45", "160770D": "46", "161102D": "47", "160488D": "48",
}


def _get_display_name(element):
    dn = element.find("iso:displayName", NS)
    if dn is not None:
        return dn.get("value", "")
    return ""


def snodent_display_to_fdi(display_name):
    dn = display_name.lower()
    if   "upper" in dn and "right" in dn: quadrant = 1
    elif "upper" in dn and "left"  in dn: quadrant = 2
    elif "lower" in dn and "left"  in dn: quadrant = 3
    elif "lower" in dn and "right" in dn: quadrant = 4
    else: return ""

    if   "central incisor" in dn: pos = 1
    elif "lateral incisor" in dn: pos = 2
    elif "canine"          in dn: pos = 3
    elif "first premolar"  in dn: pos = 4
    elif "second premolar" in dn: pos = 5
    elif "first molar"     in dn: pos = 6
    elif "second molar"    in dn: pos = 7
    elif "third molar"     in dn: pos = 8
    else: return ""

    return f"{quadrant}{pos}"


def parse_aim_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
    except Exception:
        return None

    root  = tree.getroot()
    anns  = root.find("aim:imageAnnotations", NS)
    if anns is None:
        return None
    ann = anns.find("aim:ImageAnnotation", NS)
    if ann is None:
        return None

    tooth = ""; surface = ""

    phys_coll = ann.find("aim:imagingPhysicalEntityCollection", NS)
    if phys_coll is not None:
        entity = phys_coll.find("aim:ImagingPhysicalEntity", NS)
        if entity is not None:
            char_coll = entity.find(
                "aim:imagingPhysicalEntityCharacteristicCollection", NS
            )
            if char_coll is not None:
                chars = char_coll.findall(
                    "aim:ImagingPhysicalEntityCharacteristic", NS
                )
                for ch in chars:
                    q_idx_el = ch.find("aim:questionIndex", NS)
                    q_idx    = q_idx_el.get("value", "") if q_idx_el is not None else ""

                    tc = ch.find("aim:typeCode", NS)
                    if tc is None:
                        continue
                    code    = tc.get("code", "")
                    display = _get_display_name(tc)

                    if q_idx == "0":
                        tooth = snodent_display_to_fdi(display)
                        if not tooth:
                            tooth = SNODENT_TO_FDI.get(code, "")
                    elif q_idx == "1":
                        surface = SNODENT_SURFACE_MAP.get(code, "")
                        if not surface:
                            surface = DISPLAY_NAME_TO_SURFACE.get(display, "")

    return {"tooth": tooth, "surface": surface}


# =========================================================
# VALID SURFACES
# =========================================================
VALID_SURFACES = ["Occlusal", "Mesial", "Distal", "Other"]


# =========================================================
# LOAD GROUND TRUTH FROM XML
# =========================================================
def parse_case_ground_truth(case_folder):
    gt = []
    for xml_file in sorted(Path(case_folder).glob("*.xml")):
        parsed = parse_aim_xml(str(xml_file))
        if parsed is None:
            continue
        tooth   = str(parsed.get("tooth", "Unknown"))
        surface = parsed.get("surface", "Other")
        if surface not in VALID_SURFACES:
            surface = "Other"
        gt.append({"tooth": tooth, "surface": surface})
    return gt


# =========================================================
# LOAD PREDICTION FROM PCA OUTPUT
# =========================================================
def load_prediction(case_num):
    pred_path = Path("PCA_Output") / f"case_{case_num}" / f"case_{case_num}.json"
    if not pred_path.exists():
        return []

    with open(pred_path, "r") as f:
        data = json.load(f)

    preds = []
    for t in data.get("teeth_data", []):
        tooth   = str(t.get("tooth_id", "Unknown"))
        # predicted_surface_fine (v4 field) takes priority; fallback to caries_position_detail
        surface = t.get("predicted_surface_fine", t.get("caries_position_detail", "Other"))
        if surface not in VALID_SURFACES:
            surface = "Other"
        preds.append({"tooth": tooth, "surface": surface})

    return preds


# =========================================================
# MATCH GT VS PREDICTION
# =========================================================
def match_case(gt, pred):
    pred_dict = {p["tooth"]: p["surface"] for p in pred}
    y_true, y_pred = [], []
    for g in gt:
        tooth      = g["tooth"]
        gt_surface = g["surface"]
        pred_surface = pred_dict.get(tooth, "Other")
        y_true.append(gt_surface)
        y_pred.append(pred_surface)
    return y_true, y_pred


# =========================================================
# MAIN LOOP
# =========================================================
all_y_true = []
all_y_pred = []

base_gt = Path("../data/500 cases with annotation")

for case_num in range(1, 501):
    gt_folder = base_gt / f"case {case_num}"
    gt   = parse_case_ground_truth(gt_folder)
    pred = load_prediction(case_num)

    if len(gt) == 0 and len(pred) == 0:
        continue

    yt, yp = match_case(gt, pred)
    all_y_true.extend(yt)
    all_y_pred.extend(yp)


# =========================================================
# FINAL METRICS
# =========================================================
accuracy  = accuracy_score(all_y_true, all_y_pred)
precision = precision_score(all_y_true, all_y_pred, average="macro", zero_division=0)
recall    = recall_score(   all_y_true, all_y_pred, average="macro", zero_division=0)
f1        = f1_score(       all_y_true, all_y_pred, average="macro", zero_division=0)

print("\n========== FINAL EVALUATION ==========")
print(f"Total Samples : {len(all_y_true)}")
print(f"Accuracy      : {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1 Score      : {f1:.4f}")

# =========================================================
# CONFUSION MATRIX
# =========================================================
cm    = confusion_matrix(all_y_true, all_y_pred, labels=VALID_SURFACES)
cm_df = pd.DataFrame(cm, index=VALID_SURFACES, columns=VALID_SURFACES)

print("\n========== CONFUSION MATRIX ==========")
print(cm_df.to_string())

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(all_y_true, all_y_pred, labels=VALID_SURFACES, zero_division=0))



========== FINAL EVALUATION ==========
Total Samples : 1979
Accuracy      : 0.6599
Precision     : 0.5061
Recall        : 0.4865
F1 Score      : 0.4891

========== CONFUSION MATRIX ==========
          Occlusal  Mesial  Distal  Other
Occlusal       203      96      52     37
Mesial          34     554      52     30
Distal         262      63     549     47
Other            0       0       0      0

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

    Occlusal       0.41      0.52      0.46       388
      Mesial       0.78      0.83      0.80       670
      Distal       0.84      0.60      0.70       921
       Other       0.00      0.00      0.00         0

    accuracy                           0.66      1979
   macro avg       0.51      0.49      0.49      1979
weighted avg       0.73      0.66      0.69      1979

